# Hebrew Summarization with DictaLM

This notebook runs a small qualitative evaluation of `dicta-il/dictalm2.0-instruct` on Hebrew news articles from the HeSum dataset. It loads examples, generates one-sentence summaries, and displays each prediction beside its reference summary.

## 1. Load the evaluation data

Load HeSum from Hugging Face and normalize each example to a shared `text`, `summary`, and `source` structure.

In [ ]:
import datasets


def load_hesum(max_records=None) -> list[dict]:
    ds = datasets.load_dataset("biunlp/HeSum")

    records = []
    for split in ds.values():
        for record in split:
            text = record["article"].strip()
            summary = record["summary"].strip()
            record = {"text": text, "summary": summary, "source": "hesum"}
            records.append(record)

            if max_records is not None and len(records) >= max_records:
                return records

    return records


print("Loading biunlp/HeSum from HuggingFace Hub...")
records = load_hesum()
print(f"HeSum: {len(records)} usable records")

Loading biunlp/HeSum from HuggingFace Hub...


README.md:   0%|          | 0.00/595 [00:00<?, ?B/s]

data/train-00000-of-00001-fbdae94fe9f6cb(…):   0%|          | 0.00/50.6M [00:00<?, ?B/s]

data/validation-00000-of-00001-e79054a81(…):   0%|          | 0.00/6.30M [00:00<?, ?B/s]

data/test-00000-of-00001-48d3ba328be1883(…):   0%|          | 0.00/6.39M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

## 2. Load DictaLM

Select the available device, then load the tokenizer and instruction-tuned causal language model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "dicta-il/dictalm2.0-instruct"

print(f"Loading {model_id}...")


tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
).to(device)

print("Model loaded successfully!")

Loading dicta-il/dictalm2.0-instruct...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/513k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully!


## 3. Generate summaries

Prompt the model for a short factual sentence and use deterministic decoding so repeated runs are comparable.

In [ ]:
from tqdm.notebook import tqdm

# Choose the specific records to run inference on
test_records = records[:5]

print(f"Starting inference on {len(test_records)} articles...")

predictions = []
for record in tqdm(test_records, desc="Generating Summaries"):
    article = record["text"]
    reference_summary = record["summary"]

    # Ask for a factual one-sentence summary.
    messages = [
        {"role": "user", "content": f"סכם את הכתבה הבאה למשפט אחד קצר, עובדתי ומדויק, ללא הקדמות או פרשנות.\n\nכתבה:\n{article}"}
    ]

    # Format the request with the model's chat template.
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens, excluding the prompt.
    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    generated_summary = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Keep the source, reference, and prediction together for comparison.
    result_record = {
        "text": article,
        "reference_summary": reference_summary,
        "dictalm_prediction": generated_summary
    }

    predictions.append(result_record)

Starting inference on 5 articles...


Generating Summaries:   0%|          | 0/5 [00:00<?, ?it/s]

## 4. Compare predictions

Render each article, reference summary, and model summary in a right-to-left layout for quick visual review.

In [ ]:
from IPython.display import display, HTML

# Render Hebrew text right-to-left for easier comparison.
def display_prediction(text, reference_summary, dictalm_prediction):
    html_content = f"""
    <div dir="rtl" style="text-align: right; line-height: 1.8; font-size: 14px; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; padding: 15px; border: 1px solid #ddd; border-radius: 8px; background-color: #fafafa; margin-bottom: 20px;">

        <h4 style="color: #444; margin-bottom: 5px; margin-top: 0;">📰 טקסט הכתבה:</h4>
        <div style="background-color: #fff; padding: 12px; border-radius: 6px; border: 1px solid #eee; margin-bottom: 15px;">
            {text}
        </div>

        <h4 style="color: #2e7d32; margin-bottom: 5px;">🎯 סיכום מקורי (Reference):</h4>
        <div style="background-color: #e8f5e9; padding: 12px; border-radius: 6px; border: 1px solid #c8e6c9; margin-bottom: 15px;">
            {reference_summary}
        </div>

        <h4 style="color: #1565c0; margin-bottom: 5px;">🤖 סיכום המודל (DictaLM):</h4>
        <div style="background-color: #e3f2fd; padding: 12px; border-radius: 6px; border: 1px solid #bbdefb;">
            {dictalm_prediction}
        </div>

    </div>
    """
    display(HTML(html_content))


for record in predictions:
    display_prediction(
        text=record["text"],
        reference_summary=record["reference_summary"],
        dictalm_prediction=record["dictalm_prediction"]
    )